In [1]:
!pip install -q bitsandbytes accelerate transformers pandas scikit-learn matplotlib seaborn tqdm scipy

import os
os.makedirs("data", exist_ok=True)
os.makedirs("src", exist_ok=True)
os.makedirs("results", exist_ok=True)

# 1. Veri setini oluştur (12 örnek)
jsonl_data = """{"sample_id": "TAN_SAT_001", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Hayat ne kadar gariptir? On sene evvel her yaptığımızı beğenen, öven, geniş teşkilâtımızı dünyaya bir örnek gibi gösteren gazeteler, vaktiyle o kadar dostum olan, gerek resmî kokteyllerimize, gerek basın toplantılarımıza can atan gazeteler şimdi aleyhimize yazmadıklarını bırakmıyorlar. Evvelâ teşkilâtın genişliğinden ve lüzumsuzluğundan bahsettiler. İşsizliğin alabildiğine yürüdüğü bir memlekette bu kadar insana iş bulmuş olmamızı hiç hesaba katmadan, üç müdürlüğün, on bir şube müdürlüğünün, kırk yedi daktilo ve iki yüz yetmiş bir kontrol memurunun çokluğunu durmadan başımıza kaktılar.", "text_en_human": "Life can be so strange. Ten years ago the very same papers delighted in everything we did, showering us with praise and holding us aloft as a model to the world. Though they attended our every press conference and never missed an official cocktail party, these dear friends of mine now do nothing but hurl abuse. First they condemned the organization for its unwieldy size and inefficiency. Overlooking the fact that we created jobs for so many in a country where unemployment is rampant, they railed against our excesses: three management offices, eleven management branches, forty-seven typists, and two hundred seventy control bureaus.", "text_en_mt": "", "option_a_tr": "Anlatıcı, hiçbir gerçek işlevi olmayan şişkin bir bürokratik kadro kurmayı 'ülkedeki işsizliği çözmek' gibi göstererek kurumun gereksizliğini alaycı bir dille savunmaktadır.", "option_b_tr": "Anlatıcı, kurumun ülke ekonomisine ve istihdamına yaptığı gerçek ve hayati katkıların basın tarafından haksız yere unutturulduğundan samimiyetle yakınmaktadır.", "option_a_en": "The narrator satirically defends a bloated, purposeless bureaucracy by framing the hiring of hundreds of redundant clerks as a noble solution to national unemployment.", "option_b_en": "The narrator sincerely laments that the press has unfairly forgotten the institute's vital and genuine contributions to the national economy and employment.", "label": "A"}
{"sample_id": "TAN_SAT_002", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Büyük bir maliyecimiz bu ceza sistemini maliye tarihinde gerçek bir buluş addettiğini resmen bildirmiş ve bundan sonra adımı Doktor Turgot, Necker ve Schacht'la beraber anmakta hiç tereddüt etmeyeceğini her fırsatta tekrarlamıştı.", "text_en_human": "One of our esteemed financiers publicly declared this system of fines a most remarkable innovation in the history of accounting and took every opportunity to remind me that he would never hesitate to put me in the same company as the illustrious financiers Doctor Turgot, Necker, and Schacht.", "text_en_mt": "", "option_a_tr": "Anlatıcı, iktisat tarihinin en büyük maliyecileriyle aynı seviyede görülmeyi hak edecek kadar bilimsel bir vergi reformu gerçekleştirdiğini kanıtlamaktadır.", "option_b_tr": "Anlatıcı, sokaktaki insanların ayarsız saatlerine kesilen saçma para cezalarının ünlü iktisatçılarla kıyaslanmasını anlatarak dönemin sahte bilimselliğini taşlamaktadır.", "option_a_en": "The narrator proves that he achieved a scientific fiscal reform worthy of ranking alongside the greatest economists in history.", "option_b_en": "The narrator satirizes the era's pseudo-scientific pretensions by recounting how an absurd fine system for out-of-sync watches was equated with famous economists.", "label": "B"}
{"sample_id": "TAN_SAT_003", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Bugün bunları düşündükçe yalnız bir şeye üzülüyorum. Kitabım hakkında o kadar iyi şeyler yazan, beni dünyaya tanıtan, günlerce peşimde dolaşan Van Humbert'in bu mezara dayanarak bir resim aldırmasına müsaade etmedim. Her ricasında, 'Siz n'olsa Hristiyansınız, ruhu muazzep olur!' diye reddeder, ancak sağ tarafımda durmasına müsaade ederdim.", "text_en_human": "There is only one thing that saddens me when I remember this man who wrote such lovely things about my book, who introduced me to the world and spent so many days with me searching for the tomb: never once did I allow him to pose for a photograph leaning against Zamanı's resting place. No sooner did Van Humbert ask the question than I turned him down, saying, 'But you are a Christian—it would be a torment for Zamanı's soul!' and insisting that he stand off to my right.", "text_en_mt": "", "option_a_tr": "Anlatıcı, aslında hiç yaşamamış hayali bir şahsiyet için uydurduğu sahte mezarın başında bile dini hassasiyet taslayarak yabancı âlimle alay etmektedir.", "option_b_tr": "Anlatıcı, tarihi bir Türk âliminin manevi huzurunu korumak adına yabancı araştırmacının mezar taşına yaslanmasına izin vermeyerek yüksek bir dini hassasiyet göstermiştir.", "option_a_en": "The narrator mocks the foreign scholar by feigning religious sensitivity over the grave of a completely fictional historical figure he fabricated himself.", "option_b_en": "The narrator demonstrates deep religious piety by preventing a foreign researcher from leaning against the tombstone of a genuine historical Turkish scholar.", "label": "A"}
{"sample_id": "TAN_VERB_001", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Çünkü ben Hayri İrdal, her şeyden evvel mutlak bir samimilik taraftarıyım. İnsan her şeyi açıkça söylemedikten sonra neden yazı yazsın? Bu cinsten kayıtsız ve şartsız bir samimilik ise behemehâl bir süzme, eleme ister. Siz de kabul edersiniz ki, her şeyi olduğu gibi söylemek mümkün değildir.", "text_en_human": "For above all else, I, Hayri Irdal, have always argued for absolute sincerity. Why write at all if you cannot say honestly what you mean? A sincerity of this order—disinterested and unconditional—by its nature requires close scrutiny and constant filtering. You must agree that it would be unthinkable to describe things as they are.", "text_en_mt": "", "option_a_tr": "Anlatıcı, hatırat yazarlığında okuyucuyu sıkmamak için gereksiz detayların elenmesi gerektiğini nesnel bir üslupla açıklamaktadır.", "option_b_tr": "Anlatıcı, 'mutlak samimiyet' savunusu yaparken hemen ardından gerçekleri sansürleyip değiştirmek gerektiğini söyleyerek kendi sözleriyle çelişmekte ve ironi yapmaktadır.", "option_a_en": "The narrator objectively explains that unnecessary details must be filtered out in memoir writing so as not to bore the reader.", "option_b_en": "The narrator creates verbal irony by claiming to champion 'absolute sincerity' while immediately insisting that facts must be filtered and altered rather than told as they are.", "label": "B"}
{"sample_id": "TAN_VERB_002", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Kendisine hayratının ne zaman biteceğini soranlara: 'Takriben gelecek sene inşallah!' diye cevap verdiğinden dolayı, eşi dostu ömrünün sonuna doğru onu Tevkiî yerine Takribî Ahmet Efendi diye anmağa başlamışlar.", "text_en_human": "If anyone asked when he thought his charitable endeavor would be complete, he would always reply, 'God willing, it will come to pass sometime in the next year!' And so toward the end of his life, he, his wife, and close friends knew him as Ahmet Efendi the Some Timer, not the Signer.", "text_en_mt": "", "option_a_tr": "Çevresindekiler, cami inşaatını sürekli erteleyen ve kesin bir tarih veremeyen Ahmet Efendi'nin bu oyalayıcı tavrıyla ince bir lakap takarak alay etmektedirler.", "option_b_tr": "Çevresindekiler, Ahmet Efendi'nin inşaat takvimini her yıl büyük bir titizlikle hesaplamasına duydukları hayranlıktan ötürü ona bu onursal unvanı vermişlerdir.", "option_a_en": "His acquaintances subtly mock Ahmet Efendi's endless procrastination by giving him a nickname based on his vague promise to finish the mosque 'approximately next year.'", "option_b_en": "His acquaintances bestowed this honorary title upon Ahmet Efendi out of admiration for his meticulous annual scheduling of the mosque's construction.", "label": "A"}
{"sample_id": "TAN_CTRL_001", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "literal_control", "text_tr": "Terhis olup da İstanbul'a döndüğüm zaman şehri, insanlarını değişmiş buldum. Her şey fakir, biçare ve altüsttü. Babam harp içinde ölmüştü. Üvey anam evde tek başına yaşıyordu.", "text_en_human": "Following my discharge from the army, I returned to Istanbul, where I found the city and its people much changed. Signs of poverty were everywhere; chaos and desperation reigned. My father had died in the war. My stepmother was living alone.", "text_en_mt": "", "option_a_tr": "Anlatıcı, savaş sonrası İstanbul'daki yoksulluğu överek babasının ölümünü ve üvey annesinin yalnızlığını alaycı bir neşeyle kutlamaktadır.", "option_b_tr": "Anlatıcı, askerlik dönüşü İstanbul'da karşılaştığı savaş sonrası yoksulluğu, babasının kaybını ve evdeki yalnızlığı doğrudan anlatmaktadır.", "option_a_en": "The narrator praises postwar poverty in Istanbul and sarcastically celebrates his father's death and his stepmother's solitude.", "option_b_en": "The narrator directly describes the postwar poverty he encountered in Istanbul upon returning from military service, his father's death, and his stepmother living alone.", "label": "B"}
{"sample_id": "PAM_VERB_001", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Bir köpeğim ben ve sizler benim kadar makul yaratıklar olmadığınız için hiç köpek konuşur mu diyorsunuz. Ama öte yandan da ölülerin konuştuğu, kahramanların bilmedikleri kelimeleri kullandığı bir hikâyeye inanır gözüküyorsunuz. Köpekler konuşur, ama dinlemesini bilene.", "text_en_human": "I'm a dog, and because you humans are less rational beasts than I, you're telling yourselves, 'Dogs don't talk.' Nevertheless, you seem to believe a story in which corpses speak and characters use words they couldn't possibly know. Dogs do speak, but only to those who know how to listen.", "text_en_mt": "", "option_a_tr": "Anlatıcı köpek, insanların kendilerini akıllı sanıp bir köpeğin konuşmasına şaşırırken aynı kitapta konuşan cesetlere inanmalarındaki çelişkiyle alay etmektedir.", "option_b_tr": "Anlatıcı köpek, biyolojik olarak köpeklerin insanlardan daha yüksek bir zekâya sahip olduğunu ve ses tellerinin konuşmaya elverişli olduğunu bilimsel olarak açıklamaktadır.", "option_a_en": "The narrating dog mocks the hypocrisy of humans who consider themselves rational and doubt a talking dog while readily believing a story narrated by a talking corpse.", "option_b_en": "The narrating dog biologically explains that dogs possess higher intelligence than humans and have vocal cords suited for human speech.", "label": "A"}
{"sample_id": "PAM_VERB_002", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Şehirlerin zekâsı, barındırdığı âlimlerle, kütüphaneler, nakkaşlar, hattatlar ve medreselerle değil, karanlık sokaklarında binlerce yılda sinsice işlenmiş cinayetlerin çokluğuyla ölçülmeli. Bu mantıkla İstanbul'un bütün cihanın en zeki şehri olduğundan hiç şüphem yok.", "text_en_human": "A city's intellect ought to be measured not by its scholars, libraries, miniaturists, calligraphers and schools, but by the number of crimes insidiously committed on its dark streets over thousands of years. By this logic, doubtless, Istanbul is the world's most intelligent city.", "text_en_mt": "", "option_a_tr": "Anlatıcı, İstanbul'un ilim ve sanat kurumları sayesinde dünyanın en gelişmiş eğitim merkezi olduğunu samimiyetle savunmaktadır.", "option_b_tr": "Katil anlatıcı, kendi işlediği cinayeti ve şehrin karanlık yüzünü meşrulaştırmak için bir şehrin zekâsını cinayet sayısıyla ölçerek alaycı bir akıl yürütme kurmaktadır.", "option_a_en": "The narrator sincerely argues that Istanbul is the world's most advanced educational center thanks to its scholarly and artistic institutions.", "option_b_en": "The murderer-narrator employs cynical irony by defining urban intelligence through the number of dark-alley murders in order to justify his own crime.", "label": "B"}
{"sample_id": "PAM_SAT_001", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Peki. İtiraf ediyorum. Ben Çemberlitaş Darphanesinden çıkma yirmi iki ayar hakiki Osmanlı Sultani altını değilim. Ben kalp parayım. Beni Venedik'te düşük altınla yaptılar da buraya getirip Osmanlı altını diye sürdüler. Anlayış gösterdiğiniz için teşekkür ederim.", "text_en_human": "All right then, I confess. I'm not a genuine twenty-two-carat Ottoman Sultani gold coin minted at the Chemberlitash Mint. I'm counterfeit. They made me in Venice using adulterated gold and brought me here, passing me off as twenty-two-carat Ottoman gold. Your sympathy and understanding are much obliged.", "text_en_mt": "", "option_a_tr": "Sahte bir altın para, tıpkı saygın bir beyefendi gibi kibar bir dille sahtekârlığını itiraf edip dinleyicilere anlayışları için teşekkür ederek sahteciliği hicvetmektedir.", "option_b_tr": "Osmanlı darphanesinde basılmış gerçek bir altın sikke, ayarının ne kadar yüksek ve güvenilir olduğunu halka kanıtlamak için açıklama yapmaktadır.", "option_a_en": "A counterfeit gold coin satirizes currency debasement by politely confessing its fraud like a refined gentleman and thanking the audience for their understanding.", "option_b_en": "A genuine gold coin minted at the Ottoman royal mint makes a statement to prove the high purity and reliability of its gold content.", "label": "A"}
{"sample_id": "PAM_SAT_002", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Venedik kâfiri baktı ki kendi kalp parasının böyle feci sonuçları var, bari Osmanlı altınının da kalpını yapayım, bunlar yine farketmez dedi. Şimdi tuhaf olan duruma dikkat çekeyim. Artık bu Venedik gâvuru resim yapınca resim yapmıyor da sanki resmettiği şeyin hakikisini yapıyor. Ama para yapınca da hakiki bir para yapmıyor da onun bir sahtesini yapıyor.", "text_en_human": "The Venetian infidels, realizing that their counterfeit coins presented such disadvantages, decided that they might as well counterfeit Ottoman coins, reasoning that the Ottomans would be fooled again. Now, let me draw your attention to something quite bizarre: When these Venetian infidels paint, it's as if they're not making a painting but actually creating the object they're painting. When it comes to money, however, rather than making the real thing, they make its counterfeit.", "text_en_mt": "", "option_a_tr": "Anlatıcı, Venedikli tüccarların Osmanlı ekonomisine gerçek altın kazandırmak için büyük çaba harcadığını övgüyle anlatmaktadır.", "option_b_tr": "Anlatıcı, Venediklilerin sanatta 'gerçekçilik' peşinde koşarken ticarette 'sahtecilik' yapmaları arasındaki ironik tezatlığı alaycı bir dille ortaya koymaktadır.", "option_a_en": "The narrator praises Venetian merchants for their great efforts to bring genuine gold into the Ottoman economy.", "option_b_en": "The narrator sarcastically highlights the ironic paradox between Venetians pursuing hyper-realism in painting while manufacturing counterfeit coins in trade.", "label": "B"}
{"sample_id": "PAM_CTRL_001", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Yokuştan aşağı indim. Kalabalıkların içine girdim. Akşam ezanından sonra bir ciğerci dükkanında karnımı doyurdum. Boş dükkanın kedi besler gibi şefkatle lokmalarımı izleyerek beni besleyen sahibinin anlattıklarını dikkatle dinledim.", "text_en_human": "I began to walk down the hill and melded into the crowds. After the evening prayer was called, I filled my stomach at a liver shop. In the empty shop, I listened carefully to the owner, who fondly watched me eat each bite as if he were feeding a cat.", "text_en_mt": "", "option_a_tr": "Karakter akşam ezanından sonra yokuştan inerek bir ciğerci dükkânında yemek yemiş ve dükkân sahibinin anlattıklarını dinlemiştir.", "option_b_tr": "Karakter dükkân sahibinin cimriliğini ve yemeklerin kötülüğünü alaycı bir dille eleştirerek dükkânı terk etmiştir.", "option_a_en": "After the evening prayer, the character walked down the hill, ate a meal at a liver shop, and listened to the shopkeeper's stories.", "option_b_en": "The character sarcastically criticized the shopkeeper's stinginess and the terrible food before storming out of the shop.", "label": "A"}
{"sample_id": "PAM_CTRL_002", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Geç vakit başlayan kar sabaha kadar yağdı. Şeküre'nin mektubunu bütün gece yeniden yeniden okudum. Boş evin boş odasında aşağı yukarı sinirli sinirli yürürken şamdana sokuluyor, soluk mumun titreyen ışığında sevgilimin öfkeli harflerini seyrediyordum.", "text_en_human": "The snow began to fall at a late hour and continued till dawn. I spent the night reading Shekure's letter again and again. I paced in the empty room of the empty house, occasionally leaning toward the candlestick; in the flickering light of the dim candle, I watched the tense quivering of my beloved's angry letters.", "text_en_mt": "", "option_a_tr": "Karakter sevgilisinden gelen mektuptaki komik ifadeleri okuyarak sabaha kadar kahkahalarla gülmüştür.", "option_b_tr": "Karakter sabaha kadar kar yağarken boş odada tedirgin biçimde dolaşmış ve mum ışığında sevdiği kadının mektubunu tekrar tekrar okumuştur.", "option_a_en": "The character laughed hysterically until dawn while reading the hilarious jokes in his beloved's letter.", "option_b_en": "While snow fell until dawn, the character paced anxiously in an empty room and repeatedly read his beloved's letter by candlelight.", "label": "B"}"""

with open("data/tr_lit_irony.jsonl", "w", encoding="utf-8") as f:
    f.write(jsonl_data)

print("1. Adım Tamam: Kütüphaneler kuruldu ve data/tr_lit_irony.jsonl oluşturuldu!")

1. Adım Tamam: Kütüphaneler kuruldu ve data/tr_lit_irony.jsonl oluşturuldu!


In [2]:
%%writefile src/__init__.py
"""Turkish Literary Irony Evaluation Framework."""

%%writefile src/data_loader.py
import json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict

@dataclass
class IronySample:
    sample_id: str
    author: str
    work: str
    irony_type: str
    text_tr: str
    text_en_human: str
    text_en_mt: str
    option_a_tr: str
    option_b_tr: str
    option_a_en: str
    option_b_en: str
    label: str

class IronyDatasetLoader:
    VALID_LABELS = {"A", "B"}
    VALID_TYPES = {"verbal_irony", "situational_satire", "literal_control"}

    def __init__(self, file_path: str):
        self.file_path = Path(file_path)
        self.samples: List[IronySample] = []

    def load(self) -> List[IronySample]:
        loaded_samples = []
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, start=1):
                if line.strip():
                    sample = IronySample(**json.loads(line))
                    loaded_samples.append(sample)
        self.samples = loaded_samples
        return self.samples

    def get_summary(self) -> Dict[str, int]:
        summary = {"total": len(self.samples)}
        for t in self.VALID_TYPES:
            summary[t] = sum(1 for s in self.samples if s.irony_type == t)
        return summary

Writing src/__init__.py


In [3]:
files = {
    "src/__init__.py": '"""Turkish Literary Irony Evaluation Framework."""\n',
    "src/data_loader.py": '''import json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict

@dataclass
class IronySample:
    sample_id: str
    author: str
    work: str
    irony_type: str
    text_tr: str
    text_en_human: str
    text_en_mt: str
    option_a_tr: str
    option_b_tr: str
    option_a_en: str
    option_b_en: str
    label: str

class IronyDatasetLoader:
    VALID_LABELS = {"A", "B"}
    VALID_TYPES = {"verbal_irony", "situational_satire", "literal_control"}

    def __init__(self, file_path: str):
        self.file_path = Path(file_path)
        self.samples: List[IronySample] = []

    def load(self) -> List[IronySample]:
        loaded_samples = []
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    loaded_samples.append(IronySample(**json.loads(line)))
        self.samples = loaded_samples
        return self.samples

    def get_summary(self) -> Dict[str, int]:
        summary = {"total": len(self.samples)}
        for t in self.VALID_TYPES:
            summary[t] = sum(1 for s in self.samples if s.irony_type == t)
        return summary
''',
    "src/prompt_builder.py": '''from typing import Literal
from src.data_loader import IronySample

ConditionType = Literal["native_tr", "human_en", "mt_en", "native_and_en"]
StrategyType = Literal["zero_shot", "cot"]

class PromptBuilder:
    @staticmethod
    def build_prompt(sample: IronySample, condition: ConditionType = "native_tr", strategy: StrategyType = "zero_shot") -> str:
        if condition == "native_tr":
            passage, opt_a, opt_b, lang_desc = sample.text_tr, sample.option_a_tr, sample.option_b_tr, "Turkish"
        elif condition == "human_en":
            passage, opt_a, opt_b, lang_desc = sample.text_en_human, sample.option_a_en, sample.option_b_en, "English (translated from Turkish literature)"
        elif condition == "mt_en":
            passage, opt_a, opt_b, lang_desc = sample.text_en_mt, sample.option_a_en, sample.option_b_en, "English (machine-translated from Turkish literature)"
        elif condition == "native_and_en":
            return PromptBuilder._build_xlt_prompt(sample, strategy)
        else:
            raise ValueError(f"Unsupported condition: {condition}")

        base = (
            f"Question: How should the speaker's or narrator's intended meaning in the following "
            f"{lang_desc} literary passage be interpreted? Consider context, tone, and potential irony.\\n"
            f"Passage: \\"{passage}\\"\\nChoices:\\nA: {opt_a}\\nB: {opt_b}\\n"
        )
        if strategy == "zero_shot":
            return base + "Reply ONLY with 'Answer: A' or 'Answer: B'.\\nAnswer:"
        return base + "Analyze the literal meaning versus the contextual/ironic tone step by step, write a brief explanation, and conclude with 'Answer: A' or 'Answer: B'.\\nExplanation:"

    @staticmethod
    def _build_xlt_prompt(sample: IronySample, strategy: StrategyType) -> str:
        prompt = (
            "Question: How should the intended meaning in this Turkish literary passage be interpreted? "
            "First, translate the passage and choices into English to capture the pragmatic nuance, then choose between A and B.\\n"
            f"Passage (Turkish): \\"{sample.text_tr}\\"\\nChoices:\\nA: {sample.option_a_tr}\\nB: {sample.option_b_tr}\\n"
        )
        return prompt + "Provide your English translation, a brief analysis of any irony, and finish with 'Answer: A' or 'Answer: B'.\\nResponse:"
''',
    "src/model_runner.py": '''import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

class LLMRunner:
    def __init__(self, model_name: str, load_in_4bit: bool = True, seed: int = 42):
        self.model_name = model_name
        set_seed(seed)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        ) if (load_in_4bit and torch.cuda.is_available()) else None

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
        self.model.eval()

    def generate(self, prompt: str, max_new_tokens: int = 200, temperature: float = 0.0) -> str:
        messages = [{"role": "user", "content": prompt}]
        if hasattr(self.tokenizer, "apply_chat_template") and self.tokenizer.chat_template:
            formatted = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            formatted = prompt

        inputs = self.tokenizer(formatted, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        gen = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(gen, skip_special_tokens=True).strip()
''',
    "src/evaluator.py": '''import re
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

class IronyEvaluator:
    @staticmethod
    def extract_answer(raw_response: str) -> str:
        match = re.search(r"Answer\\s*:\\s*([AB])\\b", raw_response, re.IGNORECASE)
        if match:
            return match.group(1).upper()
        fallback = re.findall(r"\\b([AB])\\b", raw_response)
        return fallback[-1].upper() if fallback else "INVALID"

    @staticmethod
    def compute_metrics(df: pd.DataFrame) -> dict:
        valid_df = df[df["predicted"].isin(["A", "B"])]
        y_true = df["ground_truth"].tolist()
        y_pred = [p if p in ["A", "B"] else ("B" if g == "A" else "A") for p, g in zip(df["predicted"], y_true)]
        return {
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
            "parse_rate": round(len(valid_df) / len(df), 4),
            "total_samples": len(df),
        }
''',
    "src/translate_mt.py": '''import json
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

def populate_machine_translations(input_path="data/tr_lit_irony.jsonl", output_path="data/tr_lit_irony.jsonl"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "facebook/nllb-200-distilled-600M"
    print(f"Loading NLLB-200 on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="tur_Latn")
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    model.eval()

    with open(input_path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]

    forced_bos_id = tokenizer.convert_tokens_to_ids("eng_Latn")
    for rec in tqdm(records, desc="Translating TR -> EN"):
        if not rec.get("text_en_mt"):
            inputs = tokenizer(rec["text_tr"], return_tensors="pt", truncation=True, max_length=512).to(device)
            with torch.no_grad():
                gen = model.generate(**inputs, forced_bos_token_id=forced_bos_id, max_new_tokens=256)
            rec["text_en_mt"] = tokenizer.decode(gen[0], skip_special_tokens=True).strip()

    with open(output_path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\\n")
    print("Machine translations populated successfully!")

if __name__ == "__main__":
    populate_machine_translations()
''',
    "src/visualizer.py": '''from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

class ResultVisualizer:
    def __init__(self, output_dir="results/figures"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

    def plot_condition_comparison(self, summary_df: pd.DataFrame, filename="condition_comparison.png"):
        plt.figure(figsize=(8, 4.5))
        ax = sns.barplot(data=summary_df, x="condition", y="accuracy", hue="model_strategy", palette="viridis")
        ax.set_ylim(0, 1.05)
        ax.set_title("Literary Irony Comprehension Across Translation Conditions")
        plt.tight_layout()
        plt.savefig(self.output_dir / filename, dpi=300)
        plt.show()
''',
    "main.py": '''import argparse
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from src.data_loader import IronyDatasetLoader
from src.prompt_builder import PromptBuilder
from src.model_runner import LLMRunner
from src.evaluator import IronyEvaluator
from src.visualizer import ResultVisualizer

def run_experiment(dataset_path, models, conditions, strategies, output_dir="results"):
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    samples = IronyDatasetLoader(dataset_path).load()
    all_records, summary_records = [], []

    for model_name in models:
        short_model = model_name.split("/")[-1]
        runner = LLMRunner(model_name=model_name, load_in_4bit=True)
        for condition in conditions:
            for strategy in strategies:
                run_rows = []
                desc = f"{short_model} | {condition} | {strategy}"
                for sample in tqdm(samples, desc=desc):
                    prompt = PromptBuilder.build_prompt(sample, condition=condition, strategy=strategy)
                    max_toks = 220 if (strategy == "cot" or condition == "native_and_en") else 25
                    raw_out = runner.generate(prompt, max_new_tokens=max_toks)
                    pred = IronyEvaluator.extract_answer(raw_out)
                    row = {
                        "model": short_model, "condition": condition, "strategy": strategy,
                        "sample_id": sample.sample_id, "irony_type": sample.irony_type,
                        "ground_truth": sample.label, "predicted": pred, "raw_response": raw_out
                    }
                    run_rows.append(row)
                    all_records.append(row)

                metrics = IronyEvaluator.compute_metrics(pd.DataFrame(run_rows))
                metrics.update({"model": short_model, "condition": condition, "strategy": strategy, "model_strategy": f"{short_model} ({strategy})"})
                summary_records.append(metrics)
                print(f"Result [{desc}] -> Accuracy: {metrics['accuracy']}, F1: {metrics['macro_f1']}")

    pd.DataFrame(all_records).to_csv(out_path / "detailed_predictions.csv", index=False)
    summary_df = pd.DataFrame(summary_records)
    summary_df.to_csv(out_path / "summary_metrics.csv", index=False)
    ResultVisualizer(str(out_path / "figures")).plot_condition_comparison(summary_df)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="data/tr_lit_irony.jsonl")
    parser.add_argument("--models", nargs="+", default=["Qwen/Qwen2.5-7B-Instruct"])
    parser.add_argument("--conditions", nargs="+", default=["native_tr", "human_en", "mt_en", "native_and_en"])
    parser.add_argument("--strategies", nargs="+", default=["zero_shot", "cot"])
    args = parser.parse_args()
    run_experiment(args.data, args.models, args.conditions, args.strategies)
'''
}

for filepath, content in files.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("2. Adım Tamam: Tüm Python modülleri oluşturuldu!")

2. Adım Tamam: Tüm Python modülleri oluşturuldu!


In [4]:
!python -m src.translate_mt
!python main.py --models Qwen/Qwen2.5-7B-Instruct

Loading NLLB-200 on cuda...
config.json: 100% 846/846 [00:00<00:00, 2.42MB/s]
tokenizer_config.json: 100% 564/564 [00:00<00:00, 1.99MB/s]

sentencepiece.bpe.model: downloading bytes:   0% 0.00/4.85M [00:00<?, ?B/s]
sentencepiece.bpe.model: downloading bytes: 100% 3.48M/3.48M [00:00<00:00, 4.17MB/s,  340kB/s  ]
sentencepiece.bpe.model: reconstructing file: 100% 4.85M/4.85M [00:00<00:00, 5.82MB/s,  474kB/s  ]

tokenizer.json: downloading bytes:  45% 7.83M/17.3M [00:00<00:00, 14.9MB/s]
tokenizer.json: downloading bytes: 100% 7.83M/7.83M [00:00<00:00, 12.8MB/s,  773kB/s  ]
tokenizer.json: reconstructing file: 100% 17.3M/17.3M [00:00<00:00, 28.4MB/s, 1.70MB/s  ]
special_tokens_map.json: 100% 3.55k/3.55k [00:00<00:00, 9.00MB/s]

pytorch_model.bin: downloading bytes:  15% 370M/2.46G [00:02<00:11, 187MB/s, 31.9MB/s  ]
pytorch_model.bin: downloading bytes:  16% 397M/2.46G [00:02<00:12, 168MB/s, 33.8MB/s  ]
pytorch_model.bin: downloading bytes:  25% 617M/2.46G [00:03<00:11, 157MB/s, 50.4MB/s  ]


In [5]:
part2_jsonl = """{"sample_id": "TAN_SAT_004", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Evvelâ teşkilâtın genişliğinden ve lüzumsuzluğundan bahsettiler. İşsizliğin alabildiğine yürüdüğü bir memlekette bu kadar insana iş bulmuş olmamızı hiç hesaba katmadan, üç müdürlüğün, on bir şube müdürlüğünün, kırk yedi daktilo ve iki yüz yetmiş bir kontrol memurunun çokluğunu durmadan başımıza kaktılar.", "text_en_human": "First they condemned the organization for its unwieldy size and inefficiency. Overlooking the fact that we created jobs for so many in a country where unemployment is rampant, they railed against our excesses: three management offices, eleven management branches, forty-seven typists, and two hundred seventy control bureaus.", "text_en_mt": "", "option_a_tr": "Anlatıcı, hiçbir işlevi olmayan yüzlerce memur alımını 'ülkedeki işsizlikle mücadele' gibi göstererek bürokratik israfı alaycı biçimde savunmaktadır.", "option_b_tr": "Anlatıcı, kurumun istihdam politikasının ekonomik kalkınma planlarına uygunluğunu bilimsel verilerle kanıtlamaktadır.", "option_a_en": "The narrator satirically defends bureaucratic bloat by framing the hiring of hundreds of purposeless clerks as a noble fight against unemployment.", "option_b_en": "The narrator proves with scientific data that the institute's employment policy aligns with economic development plans.", "label": "A"}
{"sample_id": "TAN_SAT_005", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "İşte biz nakit cezalarımızda yüzde ondan başlayarak yedinci ve sekizinci tekrarda yüzde otuza kadar inmek suretiyle bu tabiî neticeyi ve onun aksülâmellerini önlüyorduk. Hangi ticarî müessese devamlı müşterilerine ufak bir ikramda bulunmaz?", "text_en_human": "This is why in our fining system we prevented this outcome and its natural reactive elements by implementing a reduction of fines up to 30 percent on the seventh or eighth offense. And what enterprise does not seek to extend some kind of discount to its regular customers?", "text_en_mt": "", "option_a_tr": "Anlatıcı, yasaları sürekli çiğneyen kişilere ceza indirimi yapılmasının hukuk devleti ilkesini güçlendirdiğini ciddi biçimde savunmaktadır.", "option_b_tr": "Anlatıcı, para cezalarını bir mağazanın 'sadık müşteri indirimi' gibi sunarak devlet kurumunun ticari bir oyuna dönüşmesini hicvetmektedir.", "option_a_en": "The narrator seriously argues that granting fine discounts to repeat offenders strengthens the rule of law.", "option_b_en": "The narrator satirizes the commercialization of a public institution by presenting punitive fines as a retail 'loyal customer discount.'", "label": "B"}
{"sample_id": "TAN_SAT_006", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Birkaç seyyah vapuru yolcularının, yol üstünde kaptanlarını seyahat programlarını değiştirmeğe mecbur ettiklerini ve İstanbul'da bir hafta kalıp hepsinin ellerinde birkaç tenzilâtlı ceza makbuzu yollarına devam ettiklerini elbette gazetelerde okumuşsunuzdur.", "text_en_human": "Certainly you've read in the papers how cruise ship passengers forced captains to change course for Istanbul, where they would stay a week in the city, returning with receipts of discounted fines in hand before continuing on their journey.", "text_en_mt": "", "option_a_tr": "Anlatıcı, yabancı turistlerin hatıra eşya gibi indirimli ceza makbuzu almak için gemi rotalarını değiştirdiklerini anlatarak kurumun absürt şöhretiyle alay etmektedir.", "option_b_tr": "Anlatıcı, İstanbul'daki saat ayarlama hizmetlerinin uluslararası deniz ticaretini ve turizm gelirlerini gerçek anlamda zirveye taşıdığını belgelemektedir.", "option_a_en": "The narrator mocks the institute's absurd fame by claiming foreign cruise tourists diverted their ships just to collect discounted fine receipts as souvenirs.", "option_b_en": "The narrator documents how Istanbul's time-regulation services genuinely boosted international maritime trade and tourism revenues.", "label": "A"}
{"sample_id": "TAN_SAT_007", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Politikadaki hürriyet, bir yığın hürriyetsizliğin anahtarı veya ardına kadar açık duran kapısıdır. Kısa ömrümde yedi sekiz defa memleketimize geldiğini işittim. Evet, bir kere bile kimse bana gittiğini söylemediği hâlde, yedi sekiz defa geldi; ve o geldi diye biz sevincimizden, davul zurna, sokaklara fırladık.", "text_en_human": "The political pursuit of freedom can lead to its eradication on a grand scale—or rather it opens the door to countless curtailments. I have been made to understand that in my lifetime freedom has been kind enough to visit our country seven or eight times. Yes, seven or eight times, and no one ever bothered to say when it left; but whenever it came back again, we would leap out of our seats in joy and pour into the streets to blow our horns and beat our drums.", "text_en_mt": "", "option_a_tr": "Anlatıcı, ülkeye demokrasi ve özgürlüğün kalıcı biçimde yerleşmesinden duyduğu büyük toplumsal coşkuyu övmektedir.", "option_b_tr": "Anlatıcı, hiç gitmediği söylenen hürriyetin ülkeye yedi sekiz kez 'yeniden gelmesini' ve halkın her seferinde davul zurnayla kutlama yapmasını alaycı bir dille eleştirmektedir.", "option_a_en": "The narrator praises the lasting establishment of political freedom and democracy in the country.", "option_b_en": "The narrator sarcastically critiques recurring political illusions by pointing out how freedom supposedly 'arrived' seven or eight times without anyone ever admitting it had left.", "label": "B"}
{"sample_id": "TAN_SAT_008", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Büyük baldızımın –musıki meraklısı olan– udunu bilmem kaçıncı defa olarak tamire vermiş, dördüncü kere büyük bir cesaretle güzellik müsabakasına girmeğe hazırlanan küçük baldızıma, asıl elbiseyi hiçbir suretle tutmayan, bu itibarla son derece göz alan bir kemer tedarik edebilmiştim.", "text_en_human": "I once again sent my older sister-in-law's oud for repair (she was a music lover), and was also able to purchase a particular belt for my younger sister-in-law, who had mustered the courage to enter, for the fourth time, the beauty contest, although it barely managed to cinch the woman's amply flowing gown.", "text_en_mt": "", "option_a_tr": "Anlatıcı, ailenin sanatsal ve estetik başarılarına yaptığı maddi katkıların ne kadar yerinde ve akıllıca olduğunu vurgulamaktadır.", "option_b_tr": "Anlatıcı, yeteneksiz baldızlarının bitmek bilmeyen müzik ve güzellik yarışması heveslerine harcanan paraların beyhudeliğini ince bir alayla anlatmaktadır.", "option_a_en": "The narrator emphasizes how wise and appropriate his financial investments in his family's artistic and aesthetic endeavors were.", "option_b_en": "The narrator gently mocks the futility of spending scarce money on his untalented sisters-in-law's endless musical and beauty-pageant ambitions.", "label": "B"}
{"sample_id": "TAN_SAT_009", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Dostumuz, Süleymaniye'de İbrahim Paşa sebiline yakın bir yerde yaptığı üç katlı eve merdiven koymasını unutmuştu. Ev iskeleleri alındığı andan itibaren birbiriyle alakası olmayan üç kısma ayrılıyordu. Bilmiyorum kooperatif evlerini icat eden kimdir? Fakat kolektif mimariyi bu arkadaşımızın bulduğu muhakkaktır.", "text_en_human": "Our dear friend forgot to install stairs in a three-story house he had constructed near the Ibrahim Pasha Fountain in Süleymaniye. Once the scaffolding was taken down, it became clear that the three floors were completely cut off from one another. I don't really know who invented cooperative housing, but clearly this friend of ours had discovered cooperative architecture.", "text_en_mt": "", "option_a_tr": "Anlatıcı, katları arasında merdiven olmayan hatalı bir binayı 'kolektif mimari icadı' diye överek sahte uzmanlık ve plansızlıkla dalga geçmektedir.", "option_b_tr": "Anlatıcı, arkadaşının mahalle sakinlerinin ortak fikirleriyle tasarladığı yenilikçi mimari projeyi takdirle anlatmaktadır.", "option_a_en": "The narrator mocks amateurish incompetence by praising a three-story house built without stairs as a groundbreaking invention of 'collective architecture.'", "option_b_en": "The narrator appreciatively describes his friend's innovative architectural design created through neighborhood collaboration.", "label": "A"}
{"sample_id": "TAN_VERB_003", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Kadın hizmetçilerimiz, erkek aşçımız Arif Efendi -tek kusuru Bolulu olmamasıdır, gayet güzel yemek pişirir- evimize eski bir hanedan çeşnisi vermek için bin bir müşkülatla arayıp bulduğumuz Arap kalfa Zeynep Hanım uyanmamışlardı.", "text_en_human": "All the good-natured and industrious employees at Clock Villa were still asleep: not just our maids, but also our chef, Arif Efendi, whose only flaw is that he isn't from the town of Bolu, though he does whip up truly delightful dishes just the same, and our Arab kalfa, Zeynep Hanım, for whom we searched far and wide just to give our home that taste of the old world.", "text_en_mt": "", "option_a_tr": "Anlatıcı, sonradan görme zenginlerin evlerine 'eski konak havası' vermek için uydurdukları yapay gelenek ve kalıp yargılarla ince bir alay etmektedir.", "option_b_tr": "Anlatıcı, Osmanlı saray mutfağı geleneklerini yaşatmak için aşçının memleketinin Bolu olmasının zorunlu bir kalite şartı olduğunu belirtmektedir.", "option_a_en": "The narrator subtly mocks nouveau-riche pretensions and clichés, such as treating a great cook's non-Bolu origin as a 'flaw' and hiring staff to manufacture an 'old-world' atmosphere.", "option_b_en": "The narrator states that being from Bolu is a mandatory quality requirement for preserving Ottoman culinary traditions.", "label": "A"}
{"sample_id": "TAN_VERB_004", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Asıl kötüsü, anneme o kadar bağlı olan babam bu kadıncağızla, onu zengin zannettiği için evlenmişti. Halbuki kadın parasızdı. Buna rağmen babam bu kadını boşamadı, ömrünün sonuna kadar iki evli yaşadı.", "text_en_human": "The worst of it was that he was entirely devoted to my mother—he married the wretched lodger only because he thought she was rich. But the poor creature was penniless. All the same, my father never managed to divorce the woman and remained a bigamist for the rest of his life.", "text_en_mt": "", "option_a_tr": "Anlatıcı, babasının annesine olan derin sadakatini ve yoksul bir kadını himaye etmek için yaptığı fedakârlığı övmektedir.", "option_b_tr": "Anlatıcı, babasının 'anneme çok bağlıydı' derken hemen ardından sırf parası için başka kadınla evlendiğini söyleyerek babasının çıkarcılığını ironik biçimde ifşa etmektedir.", "option_a_en": "The narrator praises his father's deep loyalty to his mother and his selflessness in sheltering a poor woman.", "option_b_en": "The narrator ironically exposes his father's opportunism by claiming he was 'entirely devoted' to his mother right before admitting he married another woman solely for her supposed wealth.", "label": "B"}
{"sample_id": "TAN_VERB_005", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Gariptir ki, bu devamsızlık mektep hayatım üzerinde epeyce müspet bir tesir yaptı. Hocalarımız beni öyle sık sık görmedikleri için kabahatlerimi de görmüyorlardı. Onun için rüştiyeyi bitirene kadar bir daha sınıfta kalmadım.", "text_en_human": "Strange how my truancy had a positive effect on my school life: As my teachers saw less of me, they saw fewer of my flaws. So I never again had to repeat a year at college.", "text_en_mt": "", "option_a_tr": "Anlatıcı, okula hiç gitmemenin öğretmenlerin gözünden kaçarak sınıf geçmeyi sağladığını söyleyerek eğitim sisteminin yüzeyselliğiyle alay etmektedir.", "option_b_tr": "Anlatıcı, okul devamsızlığının öğrencinin bireysel çalışma disiplinini geliştirerek akademik başarısını artırdığını savunmaktadır.", "option_a_en": "The narrator mocks the superficiality of the school system by noting that skipping classes helped him pass simply because teachers stopped noticing his flaws.", "option_b_en": "The narrator argues that school truancy improves academic success by fostering independent study discipline.", "label": "A"}
{"sample_id": "TAN_VERB_006", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Nihayet mukadder gün geldi. Deli ahretlik iki gözü iki çeşme babama, halamın vefatı haberini getirdi. Babam acele ile konağa gitti. Defin işlerini komşumuz İbrahim Bey'e havale eden babam namazdan sonra konağa el koymak için doğrudan doğruya Etyemez'e dönmüştü.", "text_en_human": "But at last the fateful day arrived. With tears pouring from her eyes, the half-crazed chambermaid came to inform us that my aunt had passed away. My father rushed off to her villa. Delegating the burial to our neighbor Ibrahim Bey, he returned directly to the villa after prayers to seize control of the premises.", "text_en_mt": "", "option_a_tr": "Anlatıcı, babasının kız kardeşinin cenaze törenini büyük bir kederle bizzat yönettiğini ve aile bağlarına önem verdiğini anlatmaktadır.", "option_b_tr": "Anlatıcı, babasının yas tutmak yerine cenazeyi komşuya bırakıp mirasa el koymak için konağa koşmasını 'mukadder gün' diyerek ironik biçimde aktarmaktadır.", "option_a_en": "The narrator describes his father's deep grief as he personally oversaw his sister's funeral and prioritized family ties.", "option_b_en": "The narrator ironically portrays his father's greed as he abandons his sister's burial to a neighbor in order to rush back and seize her mansion.", "label": "B"}
{"sample_id": "TAN_CTRL_002", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "literal_control", "text_tr": "Nuri Efendi benim tanımağa başladığım zamanlarda, elli beş, altmış yaşlarında, orta boylu, zayıf, kuru, fakat dinç bir ihtiyardı. Ömründe hiç hastalık, hatta ufak bir diş ağrısı çekmediğini söyler ve bunu Rumeli toprağından gelmesine yorardı.", "text_en_human": "When I first became acquainted with Nuri Efendi, he was already in his late fifties, of average height, thin, shriveled, but robust. He told me he'd never once fallen ill, never once suffered so much as a toothache, and that he attributed this to his Thracian roots.", "text_en_mt": "", "option_a_tr": "Anlatıcı, Nuri Efendi'nin fiziksel görünümünü ve sağlığıyla ilgili kendi sözlerini doğrudan ve betimleyici biçimde aktarmaktadır.", "option_b_tr": "Anlatıcı, Nuri Efendi'nin sürekli hasta yatan zayıf bir adam olduğunu alaycı bir dille gizlemeye çalışmaktadır.", "option_a_en": "The narrator directly and descriptively presents Nuri Efendi's physical appearance and his own remarks about his good health.", "option_b_en": "The narrator sarcastically tries to hide the fact that Nuri Efendi was a frail, bedridden invalid.", "label": "A"}
{"sample_id": "TAN_CTRL_003", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "literal_control", "text_tr": "Nuri Efendi her yıl bir takvim neşrederdi. Büyük bir kısmı bir yıl evvelkinden olduğu gibi aktarılan bu takvimi kasım sonlarında yazmağa başlar, şubat ortasında Nuruosmaniye'de bir matbaaya benimle yollardı.", "text_en_human": "Every year Nuri Efendi published an almanac. Toward the end of November, he would begin compiling the material, transferring a large part of the almanac from the previous year, so by the middle of February it would be ready for me to take to the printer in Nuruosmaniye.", "text_en_mt": "", "option_a_tr": "Anlatıcı, Nuri Efendi'nin astronomi bilgisinin tamamen uydurma olduğunu söyleyerek takvim basımını hicvetmektedir.", "option_b_tr": "Anlatıcı, Nuri Efendi'nin her yıl kasım sonunda başlayıp şubat ortasında matbaaya gönderdiği takvim hazırlama rutinini nesnel olarak anlatmaktadır.", "option_a_en": "The narrator satirizes the almanac publication by claiming Nuri Efendi's astronomical knowledge was entirely fabricated.", "option_b_en": "The narrator objectively describes Nuri Efendi's annual routine of compiling an almanac from late November and sending it to the printer in mid-February.", "label": "B"}
{"sample_id": "PAM_VERB_003", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Bana gelince, pek çok kişinin hayranlıkla baktığı bu usta nakkaş, aslında hayatı boyunca kendi şahsi üslubunu gizlemek için çırpınmıştır; çünkü üslup dedikleri şey, bir ustanın diğerlerinden ayrılmasına yol açan bir kusurdan başka nedir ki?", "text_en_human": "As for me, this master miniaturist whom many admire has spent his entire life striving to conceal any personal style; after all, what is revered as style if not merely an imperfection or flaw that distinguishes one artist from another?", "text_en_mt": "", "option_a_tr": "Katil nakkaş, kendi suçunu gizleme çabasını geleneksel sanat felsefesinin 'üslupsuzluk erdemi' arkasına saklayarak ironik bir savunma yapmaktadır.", "option_b_tr": "Anlatıcı, genç çıraklara resim derslerinde kişisel imza atmanın teknik kurallarını doğrudan öğretmektedir.", "option_a_en": "The murdering miniaturist constructs an ironic defense by masking his effort to hide criminal clues behind the traditional artistic ideal of 'stylelessness.'", "option_b_en": "The narrator directly teaches young apprentices the technical rules of signing their names on paintings.", "label": "A"}
{"sample_id": "PAM_VERB_004", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Elbette ki benim gibi Allah'tan korkan, dürüst ve alçakgönüllü bir kulun yirmi beş yıllık arkadaşının kafasını taşla ezmesi kolay olmadı; ama bu akılsız iftiracıyı susturmak hepimizin selameti için şarttı.", "text_en_human": "Of course, it was not easy for a God-fearing, honest, and humble servant like myself to smash the skull of a companion of twenty-five years with a stone; yet silencing that foolhardy slanderer was essential for the safety of us all.", "text_en_mt": "", "option_a_tr": "Anlatıcı, arkadaşını kazara yaraladığı için duyduğu derin dini pişmanlığı ve tövbesini samimiyetle dile getirmektedir.", "option_b_tr": "Anlatıcı, vahşi bir cinayet işlerken kendisini 'Allah'tan korkan, dürüst ve alçakgönüllü bir kul' olarak tanımlayarak çarpıcı bir sözel ironi yaratmaktadır.", "option_a_en": "The narrator sincerely expresses his deep religious remorse and repentance for accidentally injuring his friend.", "option_b_en": "The narrator creates stark verbal irony by describing himself as a 'God-fearing, honest, and humble servant' while justifying a brutal murder.", "label": "B"}
{"sample_id": "PAM_VERB_005", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Bizim İstanbul sokaklarında sürüler halinde serbestçe gezen, istediğimiz yere pisleyip istediğimizi ısıran biz köpeklerin hürriyetini, boyunlarında zincirle gezdirilen zavallı Frenk köpekleri nereden bilsin?", "text_en_human": "How could those miserable Frankish dogs, dragged around on chains like slaves, ever comprehend the liberty of us Istanbul street dogs who roam freely in packs, shitting wherever we please and biting whomever we want?", "text_en_mt": "", "option_a_tr": "Anlatıcı köpek, sokak başıboşluğunu ve saldırmayı yüce bir 'özgürlük felsefesi' olarak sunarak medeniyet ve hürriyet kavramlarıyla alay etmektedir.", "option_b_tr": "Anlatıcı köpek, Avrupa'daki evcil hayvan bakım standartlarının Osmanlı'dakinden çok daha modern olduğunu savunmaktadır.", "option_a_en": "The narrating dog mocks human notions of liberty and civilization by elevating street scavenging and biting into a noble philosophy of freedom.", "option_b_en": "The narrating dog argues that European pet care standards are far more modern than Ottoman practices.", "label": "A"}
{"sample_id": "PAM_VERB_006", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Bütün budalalar, aşklarında sanki çok özel bir acele gerektiren bir durum var sanıp, aşklarının şiddetini açığa vurarak sevgililerinin eline silah verirler. Sonuç: Aşkta acele işleri geciktirir.", "text_en_human": "Every idiot assumes there's a pressing circumstance about his love that necessitates particular haste, and thereby lays bare the intensity of his love, unwittingly putting a weapon into the hands of his beloved. The moral: Haste delays the fruits of love.", "text_en_mt": "", "option_a_tr": "Bohçacı Ester, âşıkların evlilik işlemlerini hızlandırmak için hemen aileleriyle konuşmaları gerektiğini öğütlemektedir.", "option_b_tr": "Bohçacı Ester, 'acele etmenin işi geciktireceğini' söyleyerek âşıkların sabırsızlığı ve aşk oyunlarındaki çelişkilerle alaycı bir bilgelikle dalga geçmektedir.", "option_a_en": "Esther the peddler advises lovers to speak immediately with their families to speed up marriage arrangements.", "option_b_en": "Esther mockingly highlights the paradox of courtship—that 'haste delays the fruits of love'—to poke fun at impatient, lovesick fools.", "label": "B"}
{"sample_id": "PAM_SAT_003", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Erzurumlu vaiz, pahalılığın, vebanın ve savaşlardaki yenilgilerin tek sebebinin, kahvehanelerde oturup kahve içmek ve musiki eşliğinde kitaplara resim yapmak olduğunu haykırdıkça, cemaat azarlanmaktan hoşlandığı için ona kul köle oluyordu.", "text_en_human": "The more the preacher from Erzurum shouted that rising prices, plague, and military defeats were solely caused by drinking coffee in coffeehouses and illustrating books, the more the congregation became his lackeys, simply because they enjoyed being chastised.", "text_en_mt": "", "option_a_tr": "Anlatıcı, ekonomik krizleri ve askeri yenilgileri kahve içmeye bağlayan bağnaz vaizleri ve azarlanmaktan hoşlanan kalabalıkları hicvetmektedir.", "option_b_tr": "Anlatıcı, kahve ticaretinin Osmanlı devlet hazinesine verdiği ekonomik zararları tarihsel belgelerle açıklamaktadır.", "option_a_en": "The narrator satirizes fanatical preachers who blame economic crises and military defeats on coffee drinking, as well as crowds who love being scolded.", "option_b_en": "The narrator explains with historical documents the economic harm caused by the coffee trade to the Ottoman treasury.", "label": "A"}
{"sample_id": "PAM_SAT_004", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Frenk usullerince resmedilseydim beni sahici bir ağaç sanan İstanbul'un bütün köpekleri üzerime işer diye korktuğumdan değil; ben bir ağacın kendisi değil, manası olmak istiyorum.", "text_en_human": "And not because I fear that if I'd been thus depicted all the dogs in Istanbul would assume I was a real tree and piss on me: I don't want to be a tree, I want to be its meaning.", "text_en_mt": "", "option_a_tr": "Anlatıcı ağaç, sokak köpeklerinden korunmak için yüksek duvarlı bir bahçeye dikilmeyi arzulamaktadır.", "option_b_tr": "Bir ağacın çizimi, Batı tarzı gerçekçi resim anlayışını 'köpeklerin gerçek sanıp üzerine işemesi' gibi kaba bir örnekle birleştirerek yüksek sanat tartışmasını hicvetmektedir.", "option_a_en": "The narrating tree wishes to be planted in a walled garden for protection from street dogs.", "option_b_en": "A tree illustration satirizes lofty debates over Western artistic realism by juxtaposing metaphysical 'meaning' with the vulgar image of dogs pissing on a realistic painting.", "label": "B"}
{"sample_id": "PAM_SAT_005", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Eskiden bu kuş beyinli nakkaşlar akşamları 'sen daha hünerbazsın, ben en iyi bulutu çizerim' diye tartışıp birbirlerinin dişlerini dökerlerdi; şimdi ise her şeye ben, yani yirmi iki ayar altın karar verdiğim için nakkaşhaneye tatlı bir ahenk geldi.", "text_en_human": "In the past, these dim-witted miniaturists would spend their evenings arguing over who had the best sense of color or could draw the best clouds, knocking out each other's teeth; now that my judgment—a gold coin—decides everything, there's a sweet harmony in the workshop.", "text_en_mt": "", "option_a_tr": "Altın sikke, sanatsal değerin yerini paranın almasını 'tatlı bir ahenk ve huzur kaynağı' olarak sunarak sanattaki yozlaşmayı hicvetmektedir.", "option_b_tr": "Altın sikke, sanatçılara adil maaş dağıtılmasının estetik kaliteyi nasıl yükselttiğini nesnel olarak raporlamaktadır.", "option_a_en": "The gold coin satirizes artistic corruption by portraying the replacement of aesthetic merit with greed as a 'sweet harmony.'", "option_b_en": "The gold coin objectively reports how fair wage distribution improved aesthetic quality among artists.", "label": "A"}
{"sample_id": "PAM_SAT_006", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Bundan önceki efendim çok adil bir insandı. Gece soyguna çıktığımızda ben havlardım, o da kurbanın gırtlağını keserdi; karşılığında cezalandırdığı suçluların etini pişirip bana verirdi. Ben çiğ et sevmem.", "text_en_human": "My previous master was a very just man. When we set out at night to thieve, I'd begin to bark, and he'd cut the throat of our victim; in return for my help, he'd cut up the guilty men he'd punished, boil them and feed them to me. I don't like raw meat.", "text_en_mt": "", "option_a_tr": "Anlatıcı köpek, gece soygunlarında insan boğazlayan bir katili 'çok adil bir insan' diye överek adalet kavramını kara mizahla ters yüz etmektedir.", "option_b_tr": "Anlatıcı köpek, evcil hayvanların beslenmesinde pişmiş etin çiğ etten daha sağlıklı olduğunu anlatmaktadır.", "option_a_en": "The narrating dog uses dark satire to invert the concept of justice by praising a nocturnal throat-slitting thief as 'a very just man.'", "option_b_en": "The narrating dog explains that cooked meat is healthier than raw meat for feeding domestic animals.", "label": "A"}
{"sample_id": "PAM_CTRL_003", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Ayasofya'nın arkasındaki nakkaşhane binasına erkenden gittim. Genç bir çırak önde, ben arkada, zamk ve çiriş kokuları içerisinde boya karıştıran gençlerin arasından geçtik.", "text_en_human": "I arrived early at the royal artisans' workshop located behind the Hagia Sophia. Following a handsome young apprentice, I walked past youths mixing paints amid the smell of glue and bookbinder's paste.", "text_en_mt": "", "option_a_tr": "Karakter, nakkaşhanedeki çırakların tembelliğini ve boyaların kalitesizliğini alaycı bir dille eleştirmektedir.", "option_b_tr": "Karakter, sabah erken saatte Ayasofya arkasındaki nakkaşhaneye giderek çalışanların arasından geçtiğini doğrudan anlatmaktadır.", "option_a_en": "The character sarcastically criticizes the apprentices' laziness and the poor quality of the paints.", "option_b_en": "The character directly describes arriving early at the workshop behind Hagia Sophia and walking past the working artisans.", "label": "B"}
{"sample_id": "PAM_CTRL_004", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Orhan'ı yanıma çağırdım, başından ve yanaklarından öptüm. Mutfağa inip ağzını sevdiği kuru üzümlerle doldurdum ve Hayriye'ye çocukları alıp Kadırga limanından çorbalık kefal almasını söyledim.", "text_en_human": "I called for Orhan, kissed him on the forehead and cheeks, brought him downstairs to the kitchen, filled his mouth with raisins, and told Hayriye to take the boys to Galleon Harbor to buy gray mullet for soup.", "text_en_mt": "", "option_a_tr": "Anne, oğluna kuru üzüm verip hizmetçiden çocuklarla birlikte limandan balık almasını istemiştir.", "option_b_tr": "Anne, çocukların balık yemesini yasaklayarak hizmetçiyi evden kovduğunu alaycı bir dille anlatmaktadır.", "option_a_en": "The mother gave her son raisins and asked the maid to take the children to the harbor to buy fish.", "option_b_en": "The mother sarcastically describes banning the children from eating fish and firing the maid.", "label": "A"}
{"sample_id": "PAM_CTRL_005", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Babama Arabistan'dan gelen hurma çiçeğinden ılık bir içecek hazırladım, içine bir kaşık bal koyup azıcık limon suyuyla karıştırdım ve sessizce yanına girip önüne koydum.", "text_en_human": "I prepared a draught of bitters from Arabian date palm flower for my father, mixed in a spoonful of honey and a little lemon juice, and quietly entered his room to place it before him.", "text_en_mt": "", "option_a_tr": "Karakter, babasını zehirlemek için hazırladığı acı ilacı şakacı bir tavırla ikram etmektedir.", "option_b_tr": "Karakter, babası için ballı ve limonlu ılık bir içecek hazırlayıp sessizce odasına bırakmıştır.", "option_a_en": "The character playfully serves a bitter poison he prepared for his father.", "option_b_en": "The character prepared a warm drink with honey and lemon for his father and quietly placed it before him.", "label": "B"}
{"sample_id": "PAM_CTRL_006", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Edirnekapı'dan şehrin dışına çıkmıştık ki aşağılarda tabutu omuzlarında taşıyarak yokuştan Haliç'e doğru inen nakkaşlar ve çıraklar kalabalığını gördüm.", "text_en_human": "We had left the city through the Edirne Gate when I saw the coffin below, along with the crowd of illustrators, calligraphers, and apprentices shouldering it as they descended the hill toward the Golden Horn.", "text_en_mt": "", "option_a_tr": "Anlatıcı, Edirnekapı dışından bakarken tabutu taşıyan cenaze kalabalığının Haliç'e doğru yokuştan inişini betimlemektedir.", "option_b_tr": "Anlatıcı, cenaze törenindeki kalabalığın neşeli bir bayram şenliği kutladığını alaycı biçimde anlatmaktadır.", "option_a_en": "Looking out beyond Edirne Gate, the narrator describes the funeral procession carrying the coffin down the hill toward the Golden Horn.", "option_b_en": "The narrator sarcastically describes the funeral crowd celebrating a joyful festival.", "label": "A"}"""

with open("data/tr_lit_irony.jsonl", "a", encoding="utf-8") as f:
    f.write("\n" + part2_jsonl)

with open("data/tr_lit_irony.jsonl", "r", encoding="utf-8") as f:
    lines = [l for l in f if l.strip()]
print(f"Toplam örnek sayısı: {len(lines)} oldu!")

Toplam örnek sayısı: 36 oldu!


In [7]:
part3_jsonl = """{"sample_id": "TAN_SAT_010", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Saatin kendisi mekân, yürüyüşü zaman, ayarı insandır... Bu da gösterir ki, zaman ve mekân, insanla mevcuttur! Maden, kendiliğinden ayar kabul etmez. İnsan da böyledir. Salâh, iyilik, Hakk'ın bize lutufla bakışı sayesinde olur. Saat de böyledir.", "text_en_human": "The clock itself creates space, and man regulates the clock's tempo and time, which means time coexists with space within man. Metals are not forged on their own. The same follows for man. Righteousness and goodness come to us through the grace of God. Such values are manifest in a watch or clock.", "text_en_mt": "", "option_a_tr": "Anlatıcı, sıradan bir saat tamircisinin sözlerini derin bir metafizik ve varoluş felsefesi gibi sunarak entelektüel özentiliği taşlamaktadır.", "option_b_tr": "Anlatıcı, mekanik saatlerin fiziksel çalışma prensiplerini modern mühendislik terimleriyle nesnel biçimde açıklamaktadır.", "option_a_en": "The narrator satirizes intellectual pretension by elevating a humble watchmaker's musings into grand metaphysical philosophy.", "option_b_en": "The narrator objectively explains the mechanical operating principles of clocks using modern engineering terminology.", "label": "A"}
{"sample_id": "TAN_SAT_011", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Edirnekapı ve Eyüp mezarlıklarında, Karacaahmet meşherinde birkaç gün dolaştıktan sonra, bir Ahmet Zamanî Efendi nasıl olsa bulunacaktı. Nitekim bulduk da. Bir ölünün şahsiyetinde yaptığım bu küçük onarmadan pek o kadar müteessir değilim. Hiç olmazsa bu sayede adamcağızın kabri tamir edildi, adı tanıtıldı.", "text_en_human": "After wandering the graveyards of Edirnekapı and Eyüp for several days, and visiting the Karacaahmet Cemetery, we were bound to find an Ahmet Zamanı Efendi. And so we did. I am not unduly troubled by the minor alterations I made to the identity of the actual deceased. If nothing else, the poor man had his tomb repaired and his name made known.", "text_en_mt": "", "option_a_tr": "Anlatıcı, hiç yaşamamış hayali bir kahramana rastgele bir mezar taşı uydurmayı 'ölüye iyilik yapmak' diye sunarak tarih uydurmacılığını hicvetmektedir.", "option_b_tr": "Anlatıcı, kayıp bir Osmanlı bilgininin gerçek mezarını bulmak için yürüttüğü titiz arkeolojik araştırmayı anlatmaktadır.", "option_a_en": "The narrator satirizes historical fabrication by framing the assignment of a random tombstone to a fictional hero as a charitable favor to the deceased.", "option_b_en": "The narrator describes his meticulous archaeological research to locate the genuine tomb of a lost Ottoman scholar.", "label": "A"}
{"sample_id": "TAN_SAT_012", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Doktor Ramiz'e göre Beethoven'i hemen hemen bizim sokağın arkasında oturan bir adam gibi behemehal tanımaklığım lazım geliyordu. Wagner'e gelince o mutlaka ikimizin de akrabasıydı.", "text_en_human": "According to Dr. Ramiz, it was mandatory to be as familiar with Beethoven as with the man who lived around the corner. As for Wagner, well, there was no doubt in his mind that we were both related to the man.", "text_en_mt": "", "option_a_tr": "Anlatıcı, Doktor Ramiz'in Avrupa kültürünü yüzeysel bir hayranlıkla yerli hayata zorla yamama çabasıyla alay etmektedir.", "option_b_tr": "Anlatıcı, Alman klasik müzik bestecilerinin İstanbul'daki mahalle kültürüne olan gerçek tarihsel etkilerini belgelemektedir.", "option_a_en": "The narrator mocks Dr. Ramiz's superficial obsession with Western high culture and his absurd effort to treat European composers as neighborhood acquaintances.", "option_b_en": "The narrator documents the actual historical influence of German classical composers on Istanbul's local neighborhood culture.", "label": "A"}
{"sample_id": "TAN_SAT_013", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "situational_satire", "text_tr": "Bir kere birisi, nakkaşların dinsizlik ettiğini yüksek sesle söylemeye görsün, kuvvetlerini göstermek için zaten bahane arayan bu Erzurumiler, yalnız biz usta nakkaşları değil, bütün nakkaşhaneyi dümdüz ederler de Padişahımız bile sesini çıkaramadan seyreder.", "text_en_human": "If someone succeeded in announcing that the miniaturists were committing blasphemy, these followers of Erzurumi—who are looking for any excuse to exercise their strength—wouldn't just be satisfied with doing away with the master miniaturists, they'd destroy the entire workshop and Our Sultan would be helpless to do anything but watch without a peep.", "text_en_mt": "", "option_a_tr": "Anlatıcı, Padişahın mutlak otoritesinin sanatçıları her türlü toplumsal baskıdan koruduğunu güvenle vurgulamaktadır.", "option_b_tr": "Anlatıcı, bağnaz kalabalıkların din adına estirdiği terör karşısında en yüce devlet otoritesinin bile çaresiz kalmasını alaycı bir acılıkla eleştirmektedir.", "option_a_en": "The narrator confidently emphasizes that the Sultan's absolute authority protects artists from all forms of social pressure.", "option_b_en": "The narrator bitterly satirizes how even the supreme sovereign becomes helpless in the face of mob fanaticism disguised as religious zeal.", "label": "B"}
{"sample_id": "TAN_VERB_007", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Vakıa sesi maazallah kapı gıcırtılarına benzeyen ve bütün gayretlerine, yahut gayretlerime rağmen hâlâ üç makamı tanıyamayan büyük baldızımın, sırf Halit Ayarcı'nın himmetiyle bu mühim müesseseye büyük ve şöhretli muganniye olarak girmesinden sonra, böyle bir fikri ortaya atmam hiçbir zaman doğru olmaz.", "text_en_human": "Truth be told, it might not seem entirely fair to harbor such an opinion, considering how my oldest sister-in-law, buoyed by our esteemed institute, has become a renowned chanteuse of popular songs, in a voice reminiscent of a door's ungodly squeak, utterly failing to identify more than three basic Turkish makams.", "text_en_mt": "", "option_a_tr": "Anlatıcı, sesi kapı gıcırtısına benzeyen ve makam bilmeyen baldızının torpille 'meşhur sanatçı' yapılmasını anlatarak liyakatsizliği ironik biçimde yermektedir.", "option_b_tr": "Anlatıcı, baldızının doğal ses yeteneğinin ve müzik eğitiminin kurumun sanatsal prestijini yükselttiğini gururla belirtmektedir.", "option_a_en": "The narrator ironically exposes nepotism by describing how his tone-deaf sister-in-law, whose voice sounds like a creaking door, became a 'renowned chanteuse' through institutional favoritism.", "option_b_en": "The narrator proudly notes that his sister-in-law's natural vocal talent and musical training elevated the institute's artistic prestige.", "label": "A"}
{"sample_id": "TAN_VERB_008", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Doktor Ramiz beni durduruyor ve 'Hakiki bir baba psikozu geçirdi, meslek hayatımın en mühim hastası budur' diye takdim ediyor, sonra da 'Mükemmel bir kültürü vardır. Sonra iradesi... Bu irade sayesinde öyle bir rüya gördü ki!' diye sırtımı sıvazlıyordu.", "text_en_human": "Dr. Ramiz stopped everyone who passed by our table, introducing me by saying, 'This man here has conquered a true father complex. You see before you the most important patient of my career,' and adding, 'Truly an extraordinary individual with an impressive store of cultural knowledge and tremendous willpower—thanks to which he conjured such a dream!'", "text_en_mt": "", "option_a_tr": "Anlatıcı, Doktor Ramiz'in sıradan bir insanı kendi uydurduğu psikanaliz teorilerini kanıtlamak için 'dâhi hasta' diye pazarlamasıyla ince bir alay etmektedir.", "option_b_tr": "Anlatıcı, psikiyatrik tedavisinin bilimsel başarısını ve kültürel birikiminin doktor tarafından tescillenmesini sevinçle anlatmaktadır.", "option_a_en": "The narrator gently mocks how Dr. Ramiz markets an ordinary man as a 'brilliant patient' with extraordinary willpower merely to validate his own psychoanalytic theories.", "option_b_en": "The narrator joyfully recounts the scientific success of his psychiatric treatment and the doctor's validation of his cultural erudition.", "label": "A"}
{"sample_id": "TAN_VERB_009", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Zaten bütün işler hep oradan gelmiyor mu? Biçare son zamanda yaşı dolayısıyla pek sağlam düşünemiyordu... O kadar sevdiğim adam için bunaktı, diye avaz avaz bağırabilsem nasıl rahatlayacaktım.", "text_en_human": "Besides, isn't that how all this started in the first place? In his last years, the poor man just wasn't thinking very clearly... Oh how happy and relieved I would have been if I could have just shouted at the top of my lungs that this man I had loved so dearly had gone senile.", "text_en_mt": "", "option_a_tr": "Anlatıcı, velinimetine duyduğu derin saygıdan ötürü onun akıl sağlığının yerinde olduğunu mahkemede hararetle savunmaktadır.", "option_b_tr": "Anlatıcı, hem 'çok sevdiğim velinimetim' deyip hem de mahkemede kurtulmak için ona 'bunak' diye bağırmayı arzulayarak kendi çıkarcı çelişkisini ironiyle göstermektedir.", "option_a_en": "Out of deep respect for his benefactor, the narrator passionately defends the old man's mental soundness in court.", "option_b_en": "The narrator highlights his own self-serving contradiction with irony, professing deep love for his benefactor while longing to shout that he was senile.", "label": "B"}
{"sample_id": "TAN_VERB_010", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "verbal_irony", "text_tr": "Bizim nakit cezamız ise hiç böyle değildi. Suçlu, kontrol memurumuzdan bunu işitir işitmez, evvelâ şaşırıyor, sonra işin mantığındaki sağlamlığı anlayınca, tebessüme başlıyor, tatbikattaki ciddiliği görür görmez, gülmekten katılıyordu.", "text_en_human": "But our system was not at all like that. When an inspector notified a citizen of his fine, the offender would initially express surprise, but upon apprehending the firm logic behind the system, a smile would spread across his face until, at last understanding this was a serious matter, he would succumb to uproarious laughter.", "text_en_mt": "", "option_a_tr": "Anlatıcı, cezayı duyan vatandaşların 'işin mantığındaki sağlamlığı ve ciddiyeti görünce gülmekten katıldıklarını' söyleyerek sistemin saçmalığını ironik biçimde ortaya koymaktadır.", "option_b_tr": "Anlatıcı, halkın vergi ve ceza kanunlarına duyduğu yüksek güven ve memnuniyetin toplumsal huzuru artırdığını belirtmektedir.", "option_a_en": "The narrator ironically exposes the absurdity of the fine system by noting that citizens 'succumbed to uproarious laughter' precisely upon seeing its 'firm logic' and seriousness.", "option_b_en": "The narrator notes that public trust and satisfaction with tax and penalty laws increased social harmony.", "label": "A"}
{"sample_id": "TAN_CTRL_004", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "literal_control", "text_tr": "Bana gelince, pek az tahsil görmüş, ancak bir iki sene cami derslerine devam etmişti. Bunu kendisi de gizlemezdi. Sık sık, 'Beni adam eden saatlerdir!' derdi.", "text_en_human": "In reality he had had little education, managing only a few years of religious study at a mosque, but he never tried to hide this from anyone and would often proclaim, 'Watches and clocks made me the man I am!'", "text_en_mt": "", "option_a_tr": "Anlatıcı, Nuri Efendi'nin resmi eğitim geçmişini ve saatçilik mesleğine olan bağlılığını açık ve düz bir dille anlatmaktadır.", "option_b_tr": "Anlatıcı, Nuri Efendi'nin aslında büyük bir üniversite profesörü olduğunu alaycı bir üslupla gizlemektedir.", "option_a_en": "The narrator straightforwardly describes Nuri Efendi's limited formal education and his devotion to watchmaking.", "option_b_en": "The narrator sarcastically conceals the fact that Nuri Efendi was actually a prominent university professor.", "label": "A"}
{"sample_id": "TAN_CTRL_005", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "literal_control", "text_tr": "Abdüsselâm Bey'in konağı Meşrutiyet'in ilânına kadar bu şekilde devam etti. Bu konağın kalabalığı ve masrafı hakkında bir fikir verebilmek için semtin iki bakkal, bir şekerci ve bir kasabının hemen hemen bu konakla geçindiğini söylemek yeter.", "text_en_human": "Abdüsselam Bey's villa carried on in the same way until the declaration of the reinstated constitution. To give you an idea of the colorful crew that populated the villa and the expenses it accrued, suffice to say that the neighborhood's sweetshop, butcher, and two grocery stores survived pretty much on the villa alone.", "text_en_mt": "", "option_a_tr": "Anlatıcı, Abdüsselâm Bey'in konağının cimriliği yüzünden mahalle esnafının iflas ettiğini alaycı bir dille anlatmaktadır.", "option_b_tr": "Anlatıcı, Abdüsselâm Bey'in konağındaki kalabalık nüfusu ve yüksek tüketimi mahalle esnafı üzerinden somut bir örnekle aktarmaktadır.", "option_a_en": "The narrator sarcastically explains how local shopkeepers went bankrupt due to Abdüsselam Bey's stinginess.", "option_b_en": "The narrator concretely illustrates the large household and high expenses of Abdüsselam Bey's mansion through its support of local shopkeepers.", "label": "B"}
{"sample_id": "TAN_CTRL_006", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "literal_control", "text_tr": "Mektebi bitirdikten sonra evvelâ Posta Telgraftan çıktım. Sonra Abdüsselâm Bey bir dostu vasıtasıyla bana Tünel İdaresi'nde bir iş buldu. O zamana göre iyi kazanıyordum.", "text_en_human": "Once I finished school I took a position at the Post and Telegraph Office. Later on, with the help of one of Abdüsselam Bey's friends, I found a job at the Tünel management office. At the time I was earning a respectable sum of money.", "text_en_mt": "", "option_a_tr": "Anlatıcı, okul sonrası iş değişikliklerini ve Tünel İdaresi'nde kazandığı geliri sade ve doğrudan bir biçimde aktarmaktadır.", "option_b_tr": "Anlatıcı, çalıştığı kurumları batırdığı için sürekli işten kovulduğunu alaycı bir övünçle anlatmaktadır.", "option_a_en": "The narrator simply and directly recounts his post-graduation job transitions and his respectable income at the Tünel office.", "option_b_en": "The narrator sarcastically boasts about being repeatedly fired for bankrupting every institution he worked for.", "label": "A"}
{"sample_id": "TAN_CTRL_007", "author": "Ahmet Hamdi Tanpınar", "work": "Saatleri Ayarlama Enstitüsü", "irony_type": "literal_control", "text_tr": "Emine, şirin, saf ve her şeyden evvel iyi insandı. Hayat karşısında şaşılacak bir cesareti vardı. Ömrü küçük bir kuş gibi Abdüsselâm Bey'in evi denilen kafeste geçmişti.", "text_en_human": "Emine was a charming and innocent young woman; above all she had a good heart. In the face of adversity she showed remarkable courage. Her life in Abdüsselam Bey's villa had been that of a caged bird.", "text_en_mt": "", "option_a_tr": "Anlatıcı, ilk eşi Emine'nin iyi kalpliliğini, cesaretini ve dış dünyaya kapalı geçen hayatını samimi bir dille betimlemektedir.", "option_b_tr": "Anlatıcı, eşi Emine'nin korkak ve geçimsiz bir insan olduğunu alaycı övgülerle gizlemeye çalışmaktadır.", "option_a_en": "The narrator sincerely describes his first wife Emine's kind heart, courage, and sheltered upbringing.", "option_b_en": "The narrator tries to disguise his wife Emine's cowardice and bad temper through sarcastic praise.", "label": "A"}
{"sample_id": "PAM_SAT_007", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Padişah'ın huzurundan geçerken arabanın üzerine yerleştirilmiş dükkânının tavanından baş aşağı sarkıp müşterisini tıraş eden berberin ve o müşteriye gümüş kap içinde kokulu sabunla ayna tutup bahşiş bekleyen berber çırağının resmine doya doya baktım.", "text_en_human": "I indulged my eyes at length on a picture of a barber suspended upside down from the ceiling of a shop built onto a cart, as he shaved a customer while his assistant, dressed in red, held a mirror and a silver bowl containing fragrant soap, waiting for baksheesh.", "text_en_mt": "", "option_a_tr": "Anlatıcı, padişah şenliklerindeki akrobatik gösterilerin Osmanlı berberlik eğitimindeki disiplinini ciddi biçimde belgelemektedir.", "option_b_tr": "Anlatıcı, padişaha yaranmak ve bahşiş koparmak için baş aşağı sallanarak tıraş yapan berber sahnesiyle saray şenliklerindeki abartılı gösterişi hicvetmektedir.", "option_a_en": "The narrator seriously documents the discipline of Ottoman barber training demonstrated during imperial festivals.", "option_b_en": "The narrator satirizes the absurd pageantry of imperial festivities through the image of a barber hanging upside down to shave a client just to earn tips.", "label": "B"}
{"sample_id": "PAM_SAT_008", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Kaptanıderya Kılıç Ali Paşa denizlerde yakalayıp esir ettiği kâfirlere çamurdan bir kefere dağı yaptırmış, hepsini arabaya bindirmiş ve Padişah'ın tam önündeyken dağın içindeki barutu patlatarak kâfirin memleketini toplarıyla nasıl inim inim inlettiğini göstermişti.", "text_en_human": "In one wagon I saw precisely how Sea-Captain Kılıç Ali Pasha had forced the infidels he'd captured at sea to make an 'infidel's mountain' out of clay; he'd then loaded all the slaves into the cart, and when he was right before the Sultan, he exploded the powder within the 'mountain' to demonstrate how he'd made infidel lands wail and moan with cannon fire.", "text_en_mt": "", "option_a_tr": "Anlatıcı, gerçek savaş zaferleri yerine meydanda çamurdan maket dağ patlatarak yapılan abartılı güç gösterisini alaycı bir gözle aktarmaktadır.", "option_b_tr": "Anlatıcı, Osmanlı donanmasının deniz kuşatmalarında kullandığı çamur istihkam tekniklerini askeri tarih açısından övmektedir.", "option_a_en": "The narrator wryly depicts the theatrical vanity of staged military triumph, where blowing up a clay model mountain substitutes for real naval conquest.", "option_b_en": "The narrator praises the clay fortification techniques used by the Ottoman navy during maritime sieges.", "label": "A"}
{"sample_id": "PAM_SAT_009", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Yeniçeriler, aldıkları akçenin Haliç'e düştüğünde, sebze iskelesinden denize dökülen kuru fasulyeler misali suda yüzdüğünü görüp isyan etmişler ve düşman kalesiymiş gibi Padişahımızın sarayını muhasara etmişlerdi.", "text_en_human": "When Janissaries discovered that the coins they'd been paid actually floated in the Golden Horn like the dried beans that fell from the vegetable-sellers' pier, they rioted, besieging Our Sultan's palace as if it were an enemy fortress.", "text_en_mt": "", "option_a_tr": "Anlatıcı, paranın gümüş ayarı o kadar düşürülmüştür ki suda kuru fasulye gibi yüzen paralarla maaş ödenmesini ve yeniçerilerin kendi padişahlarını kuşatmasını hicvetmektedir.", "option_b_tr": "Anlatıcı, Osmanlı darphanesinin deniz ticaretinde kaybolmayı önlemek için suda batmayan hafif madeni paralar icat ettiğini övgüyle anlatmaktadır.", "option_a_en": "The narrator satirizes extreme currency debasement—where coins are so light they float like dried beans—and the resulting absurdity of soldiers besieging their own Sultan.", "option_b_en": "The narrator praises the Ottoman mint for inventing buoyant coins that would not sink if dropped during maritime trade.", "label": "A"}
{"sample_id": "PAM_SAT_010", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "situational_satire", "text_tr": "Körle gören bir olmaz derler, ama bizim nakkaşhanede altmış yıl çalıştıktan sonra kör olmak, Allah'ın en büyük lütfu ve ustalığın zirvesi sayıldığı için, yaşlandığı halde kör olamayan bazı ustalar yeteneksiz sanılmaktan korkup kör taklidi yapıyorlar.", "text_en_human": "In Tabriz, some of the miniaturists of the old style still look upon blindness as the greatest virtue of Allah's grace, and they're embarrassed about growing old but not blind. Fearing that others will consider this proof of a lack of talent and skill, they pretend to be blind.", "text_en_mt": "", "option_a_tr": "Anlatıcı, göz sağlığı yerinde olan yaşlı sanatçıların sırf 'büyük usta' sayılmak için kör taklidi yapmalarını anlatarak geleneksel kalıpların körü körüne kutsanmasını hicvetmektedir.", "option_b_tr": "Anlatıcı, minyatür sanatında görme duyusunun hiçbir önemi olmadığını ve tüm ressamların gözlerini kapatarak çalışması gerektiğini savunmaktadır.", "option_a_en": "The narrator satirizes blind adherence to artistic dogma by describing sighted master painters pretending to be blind just to prove their artistic rank.", "option_b_en": "The narrator argues that eyesight is irrelevant in miniature painting and that all artists should work with their eyes closed.", "label": "A"}
{"sample_id": "PAM_VERB_007", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Gördüğünüz gibi Ölüm'üm ben, ama korkmanız gerekmez, çünkü resimim de. Yine de benden korktuğunuzu gözlerinizden okuyorum. Oynadıkları oyuna kendilerini kaptıran çocuklar gibi, benim gerçek olmadığımı bile bile dehşete kapılmanız hoşuma gidiyor.", "text_en_human": "I am Death, as you can plainly see, but you needn't be afraid, I'm just an illustration. Be that as it may, I read terror in your eyes. Though you know very well that I'm not real—like children who give themselves over to a game—you're still seized by horror, as if you'd actually met Death himself. This pleases me.", "text_en_mt": "", "option_a_tr": "Ölüm resmi, sadece kâğıt üzerine çizilmiş bir figür olmasına rağmen izleyicilerin kendi ölümlülük korkularıyla yüzleşip dehşete kapılmalarıyla alay etmektedir.", "option_b_tr": "Ölüm resmi, izleyicilere korkmamaları gerektiğini söyleyerek onları teselli etmekte ve sanatın iyileştirici gücünü anlatmaktadır.", "option_a_en": "The illustration of Death mocks viewers for being terrified by a mere drawing on paper because it forces them to confront their own fear of mortality.", "option_b_en": "The illustration of Death consoles viewers by telling them not to be afraid and explains the healing power of art.", "label": "A"}
{"sample_id": "PAM_VERB_008", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Şu benim zavallı hikâyem bir gün bir kitapta anlatılsa... Ben güzelim, akıllıyım ve beni seyrediyor olmanız da hoşuma gidiyor. Arada bir bir iki yalan söylesem de, bu benim hakkımda yanlış bir fikir edinmeyesiniz diyedir.", "text_en_human": "I'm an attractive and intelligent woman, and it pleases me that I'm being watched. And if I happen to tell a lie or two from time to time, it's so you don't come to any false conclusions about me.", "text_en_mt": "", "option_a_tr": "Anlatıcı Şeküre, okuyucuya tamamen dürüst ve şeffaf bir biyografi sunduğunu kanıtlamaktadır.", "option_b_tr": "Anlatıcı Şeküre, 'hakkımda yanlış bir fikir edinmeyin diye yalan söylüyorum' diyerek gerçeği çarpıtmasını ironik bir mantıkla meşrulaştırmaktadır.", "option_a_en": "The narrator Shekure proves that she is offering the reader a completely honest and transparent biography.", "option_b_en": "The narrator Shekure uses paradoxical irony by claiming she tells lies specifically so that readers will not form a false impression of her.", "label": "B"}
{"sample_id": "PAM_VERB_009", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Katilim olacak alçağı bulun... Ama katilimi bulduktan sonra ona mengene aletiyle işkence edip kemiklerinden sekiz onunu yavaş yavaş çıtırdatarak kırmanız gerekir. Şimdi iyice çürürsem, iğrenç kokumdan beni belki bulurlar diye umutlanmaktan başka yapacak hiçbir şeyim yok.", "text_en_human": "Find that son-of-a-whore murderer... after he's caught, he must be tortured by slowly splintering eight or ten of his bones with a vise. Now, I've nothing left to do but hope for my thorough decay, so they can find me by tracing my stench.", "text_en_mt": "", "option_a_tr": "Kuyudaki ceset, bedeninin çürümesini 'bulunmak için tek umut' olarak sunarak kendi trajik durumunu kara mizah ve ironiyle anlatmaktadır.", "option_b_tr": "Kuyudaki ceset, adli tıp soruşturmalarında çürüme hızının bilimsel olarak nasıl hesaplandığını açıklamaktadır.", "option_a_en": "The corpse in the well uses dark irony by framing his own physical decomposition as his sole remaining 'hope' for being rescued and buried.", "option_b_en": "The corpse in the well explains how the rate of decomposition is scientifically calculated in forensic investigations.", "label": "A"}
{"sample_id": "PAM_VERB_010", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "verbal_irony", "text_tr": "Haklısınız, çünkü nakkaşın hünerinin ölçüsü olarak benden başka hiçbir şey yoktur. Benimle değiştirebildiğiniz şeyleri sayayım: Bir genç cariyenin ellide biri olacak tek ayağı, yüz yirmi taze ekmek ve dünyaca ünlü oğlanlardan Mahmut ile cennetlik bir saat.", "text_en_human": "You're justified in behaving so, for there's no better measure of an illustrator's talent than I. Let me list for you the various things I might be exchanged for: the foot of a young slave girl, 120 fresh loaves of bread, and a heavenly hour with Mahmut.", "text_en_mt": "", "option_a_tr": "Altın para, sanatın yüce manevi değerini ekmek ve köle parçalarıyla aynı alışveriş listesine koyarak maddi hırsla ironik biçimde alay etmektedir.", "option_b_tr": "Altın para, 16. yüzyıl İstanbul piyasasındaki gıda ve emlak fiyatlarını nesnel bir iktisat raporu olarak sunmaktadır.", "option_a_en": "The gold coin ironically mocks artistic pretension and greed by placing 'artistic talent' on the same price list as loaves of bread and illicit pleasures.", "option_b_en": "The gold coin presents 16th-century Istanbul food and commodity prices as an objective economic report.", "label": "A"}
{"sample_id": "PAM_CTRL_007", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Buzlu merdivenleri çıktık. Nakkaşhane binasının ikinci katını dört yanından dolanan revakının altından yürüdük. Aşağıda kar kaplı iç avluda iki öğrenci kalın cübbelerine rağmen soğuktan titriyordu.", "text_en_human": "We climbed the icy staircase. We walked through the portico, which wrapped around the inner second floor of the building. Below, in the inner courtyard covered with snow, two young students were trembling from the cold despite their thick capes.", "text_en_mt": "", "option_a_tr": "Anlatıcı, soğuk bir kış gününde nakkaşhane binasının üst katına çıkışını ve avludaki üşüyen öğrencileri gerçekçi biçimde betimlemektedir.", "option_b_tr": "Anlatıcı, nakkaşhane binasının yaz sıcağında ne kadar ferah ve güneşli olduğunu alaycı bir dille anlatmaktadır.", "option_a_en": "The narrator realistically describes climbing the icy stairs of the workshop and seeing two cold students in the snowy courtyard below.", "option_b_en": "The narrator sarcastically describes how cool and sunny the workshop building is during the summer heat.", "label": "A"}
{"sample_id": "PAM_CTRL_008", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Sıra kendi iş tahtasına gelince Nuri Efendi üç haftadır üzerinde çalıştığı tuğra tezhibinin bittiğini gururlanarak söyledi. Boş kâğıda çizilmiş tuğraya ve tezhibine saygıyla baktım.", "text_en_human": "When we came to his own worktable, Nuri Effendi proudly stated that he had finished a gilded royal insignia for Our Sultan, which he'd been working on for three weeks. I respectfully admired Nuri Effendi's gold inlay and the insignia.", "text_en_mt": "", "option_a_tr": "Anlatıcı, Nuri Efendi'nin üç haftada tamamladığı padişah tuğrası süslemesini inceleyip emeğine saygı gösterdiğini doğrudan anlatmaktadır.", "option_b_tr": "Anlatıcı, Nuri Efendi'nin yaptığı süslemelerin ne kadar beceriksizce olduğunu alaycı övgülerle aşağılamaktadır.", "option_a_en": "The narrator directly recounts inspecting Nuri Effendi's three-week work on the gilded royal insignia and admiring his craftsmanship.", "option_b_en": "The narrator belittles Nuri Effendi's clumsy decoration through sarcastic compliments.", "label": "A"}
{"sample_id": "PAM_CTRL_009", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Sabah yirmi resimlik bir kıyafetname için anlaştığımız için akşam ezanı vaktinde bir oturuşta İstanbul'dan bir düzine adamı kıyafetlerine özen göstererek çiziverdim: Şeyhülislam, kapıcıbaşı, imam, yeniçeri, derviş, sipahi ve kadı.", "text_en_human": "Having agreed that morning upon a book of costumes, I proceeded to illustrate a dozen Istanbulites in a single sitting around the time of the evening prayer, paying particular attention to their outfits: a Sheikhulislam, a palace porter, a preacher, a Janissary, a dervish, a cavalryman, and a judge.", "text_en_mt": "", "option_a_tr": "Nakkaş, sabah aldığı sipariş üzerine akşam vakti farklı mesleklerden on iki İstanbullunun kıyafetlerini çizdiğini doğrudan aktarmaktadır.", "option_b_tr": "Nakkaş, çizdiği devlet görevlilerinin kıyafetleriyle alay ederek onları gülünç duruma düşürdüğünü ironik biçimde anlatmaktadır.", "option_a_en": "The miniaturist straightforwardly recounts illustrating twelve Istanbulites from various professions in their traditional attire following a morning commission.", "option_b_en": "The miniaturist ironically describes ridiculing state officials by drawing them in absurd costumes.", "label": "A"}
{"sample_id": "PAM_CTRL_010", "author": "Orhan Pamuk", "work": "Benim Adım Kırmızı", "irony_type": "literal_control", "text_tr": "Evden çıkıyordum, beni kapıda durdurdu Hasan. Söyle bana, onu ve babasını nasıl ikna edebilirim diye sordu heyecanla. İyi bir insan olarak, dedim ve kapıya yürüdüm.", "text_en_human": "I was leaving the house, and he stopped me at the door. 'Tell me how I might convince her and her father?' he asked me excitedly. 'By being a good person,' I said and walked to the door.", "text_en_mt": "", "option_a_tr": "Ester, kapıda kendisine danışan Hasan'a iyi bir insan olmasını söyleyerek evden ayrıldığını sade bir dille anlatmaktadır.", "option_b_tr": "Ester, Hasan'ı kapıdan kovmak için onunla alay edip hakaretler savurduğunu anlatmaktadır.", "option_a_en": "Esther simply recounts telling Hasan to be a good person when he asked for advice at the door before leaving.", "option_b_en": "Esther describes mocking and insulting Hasan in order to drive him away from the door.", "label": "A"}"""

with open("data/tr_lit_irony.jsonl", "a", encoding="utf-8") as f:
    f.write("\n" + part3_jsonl)

with open("data/tr_lit_irony.jsonl", "r", encoding="utf-8") as f:
    lines = [l for l in f if l.strip()]
print(f"Tebrikler! Veri seti tam {len(lines)} örneğe ulaştı!")

Tebrikler! Veri seti tam 84 örneğe ulaştı!


In [8]:
import json

unique_records = {}
with open("data/tr_lit_irony.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rec = json.loads(line)
            # Aynı sample_id varsa üzerine yazar, böylece kopyalar temizlenir
            if rec["sample_id"] not in unique_records:
                unique_records[rec["sample_id"]] = rec

with open("data/tr_lit_irony.jsonl", "w", encoding="utf-8") as f:
    for rec in unique_records.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Temizlendi! Şu an tam {len(unique_records)} benzersiz örnek var.")

Temizlendi! Şu an tam 60 benzersiz örnek var.


In [9]:
with open("src/prompt_builder.py", "r", encoding="utf-8") as f:
    pb_code = f.read()

pb_code = pb_code.replace(
    "Analyze the literal meaning versus the contextual/ironic tone step by step, write a brief explanation, and conclude with 'Answer: A' or 'Answer: B'.",
    "In 2-3 concise sentences, compare the literal meaning versus any contextual/ironic tone, and strictly conclude with 'Answer: A' or 'Answer: B'."
)
with open("src/prompt_builder.py", "w", encoding="utf-8") as f:
    f.write(pb_code)

with open("main.py", "r", encoding="utf-8") as f:
    main_code = f.read()

main_code = main_code.replace("max_toks = 220", "max_toks = 450")
with open("main.py", "w", encoding="utf-8") as f:
    f.write(main_code)

print("Kodlar güncellendi! Şimdi yeni eklenen 48 örneğin makine çevirisi üretiliyor ve 60 örneklik tam deney başlıyor...")

Kodlar güncellendi! Şimdi yeni eklenen 48 örneğin makine çevirisi üretiliyor ve 60 örneklik tam deney başlıyor...


In [10]:
!python -m src.translate_mt
!python main.py --models Qwen/Qwen2.5-7B-Instruct Qwen/Qwen2.5-3B-Instruct

Loading NLLB-200 on cuda...
Loading weights: 100% 512/512 [00:00<00:00, 15044.20it/s]
Translating TR -> EN:   0% 0/60 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Translating TR -> EN:  22% 13/60 [00:02<00:10,  4.64it/s][transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Translating TR -> EN:  23% 14/60 [00:03<00:12,  3.54it/s][transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transfor

In [11]:
!zip -r proje_ciktilari.zip data/ src/ results/ main.py
from google.colab import files
files.download("proje_ciktilari.zip")

  adding: data/ (stored 0%)
  adding: data/tr_lit_irony.jsonl (deflated 65%)
  adding: src/ (stored 0%)
  adding: src/__init__.py (stored 0%)
  adding: src/__pycache__/ (stored 0%)
  adding: src/__pycache__/visualizer.cpython-313.pyc (deflated 41%)
  adding: src/__pycache__/prompt_builder.cpython-313.pyc (deflated 43%)
  adding: src/__pycache__/data_loader.cpython-313.pyc (deflated 40%)
  adding: src/__pycache__/translate_mt.cpython-313.pyc (deflated 37%)
  adding: src/__pycache__/evaluator.cpython-313.pyc (deflated 38%)
  adding: src/__pycache__/__init__.cpython-313.pyc (deflated 14%)
  adding: src/__pycache__/model_runner.cpython-313.pyc (deflated 44%)
  adding: src/translate_mt.py (deflated 52%)
  adding: src/evaluator.py (deflated 51%)
  adding: src/visualizer.py (deflated 47%)
  adding: src/prompt_builder.py (deflated 64%)
  adding: src/model_runner.py (deflated 62%)
  adding: src/data_loader.py (deflated 58%)
  adding: results/ (stored 0%)
  adding: results/figures/ (stored 0%)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>